# 🧠 Yuzu Memory Rebuild

Rebuild memory layer with GPU acceleration!

## Setup

In [ ]:
# Install dependencies
!pip install -q duckdb onnxruntime-gpu huggingface-hub psycopg2-binary rich

In [ ]:
import os
import duckdb
import psycopg2
import numpy as np
from datetime import datetime
from rich.console import Console

console = Console()

# Get secrets from Colab
from google.colab import userdata

SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_KEY')
HF_TOKEN = userdata.get('HF_TOKEN', '')

console.print(f"[green]✅ Loaded secrets[/green]")

## Phase 1: Export from Supabase

In [ ]:
# Connect to Supabase
conn = psycopg2.connect(SUPABASE_URL)
cur = conn.cursor()

# Get user
cur.execute("SELECT id, email, display_name, partner_name FROM users LIMIT 1")
user = cur.fetchone()
user_id = user[0]
console.print(f"👤 User: {user[2]} (ID: {user_id})")

# Get sessions
cur.execute("SELECT id, title, created_at FROM chat_sessions WHERE user_id = %s", (user_id,))
sessions = cur.fetchall()
console.print(f"📚 Sessions: {len(sessions)}")

# Get messages
cur.execute("""
  SELECT m.id, m.session_id, m.role, m.content, m.created_at
  FROM messages m
  JOIN chat_sessions cs ON m.session_id = cs.id
  WHERE cs.user_id = %s
  ORDER BY m.created_at
""", (user_id,))
messages = cur.fetchall()
console.print(f"💬 Messages: {len(messages)}")

In [ ]:
# Save to DuckDB
duck = duckdb.connect('yuzu_local.duckdb')

# Create tables
duck.execute("""
  CREATE TABLE IF NOT EXISTS messages_export (
    id INTEGER, session_id INTEGER, role VARCHAR, content TEXT, created_at TIMESTAMP
  )
")
duck.execute("""
  CREATE TABLE IF NOT EXISTS memories_with_embeddings (
    id INTEGER, user_id INTEGER, session_id INTEGER, content TEXT,
    embedding FLOAT[], memory_type VARCHAR, importance INTEGER, created_at TIMESTAMP
  )
")

# Insert sessions
for s in sessions:
  duck.execute("INSERT INTO sessions_export VALUES (?, ?, ?)", s)

# Insert messages
for m in messages:
  duck.execute("INSERT INTO messages_export VALUES (?, ?, ?, ?, ?)", m)

console.print(f"✅ Saved to DuckDB: {len(messages)} messages")

## Phase 2: Generate Embeddings (GPU)

In [ ]:
from huggingface_hub import hf_hub_download
import onnxruntime as ort

MODEL_NAME = "BAAI/bge-m3"
EMBEDDING_DIM = 1024

# Download model
console.print("📥 Downloading model...")
model_path = hf_hub_download(repo_id=MODEL_NAME, filename="onnx/model.onnx", token=HF_TOKEN)
tokenizer_path = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer.json", token=HF_TOKEN)
console.print("✅ Model downloaded!")

# Load ONNX with GPU
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
session = ort.InferenceSession(model_path, providers=providers)
console.print(f"🖥️  Provider: {session.get_providers()[0]}")

In [ ]:
import json
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

def embed_texts(texts):
    """Generate embeddings with GPU"""
    batch = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='np')
    input_ids = batch['input_ids'].astype(np.int64)
    attention_mask = batch['attention_mask'].astype(np.int64)
    
    outputs = session.run(None, {
        'input_ids': input_ids,
        'attention_mask': attention_mask
    })
    
    # Mean pooling
    embeddings = outputs[0]
    mask = attention_mask[..., np.newaxis].astype(bool)
    embeddings = embeddings.masked_fill(~mask, 0)
    sums = embeddings.sum(axis=1)
    counts = mask.sum(axis=1)
    embeddings = sums / counts
    
    # Normalize
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings = embeddings / norms
    
    return embeddings.tolist()

console.print("✅ Embedding function ready!")

In [ ]:
# Process messages
messages = duck.execute("SELECT * FROM messages_export").fetchall()
total = len(messages)
console.print(f"🔄 Processing {total} messages...")

BATCH_SIZE = 32
processed = 0

for i in range(0, total, BATCH_SIZE):
    batch = messages[i:i+BATCH_SIZE]
    texts = [m[3] for m in batch if m[3] and len(m[3]) > 10]
    
    if texts:
        embeddings = embed_texts(texts)
        
        emb_idx = 0
        for m in batch:
            if m[3] and len(m[3]) > 10:
                emb = embeddings[emb_idx]
                emb_idx += 1
                
                duck.execute("""
                  INSERT INTO memories_with_embeddings 
                  VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                """, [m[0], user_id, m[1], m[3], emb, 'episodic', 50, m[4]])
    
    processed += len(batch)
    console.print(f"  Progress: {processed}/{total}")

console.print(f"✅ Embedded: {processed} memories")

## Phase 3: Migrate to Supabase

In [ ]:
# Get memories from DuckDB
memories = duck.execute("SELECT * FROM memories_with_embeddings").fetchall()
console.print(f"📦 Ready to migrate: {len(memories)} memories")

# Migrate in batches
BATCH_SIZE = 100
inserted = 0

for i in range(0, len(memories), BATCH_SIZE):
    batch = memories[i:i+BATCH_SIZE]
    
    args = []
    for m in batch:
        emb_str = '[' + ','.join(map(str, m[4])) + ']'
        args.extend([m[1], m[2], m[3], emb_str, m[5], m[6]])
    
    values = ','.join(['(%s,%s,%s,%s::vector(1024),%s,%s)'] * len(batch))
    
    cur.execute(f"""
      INSERT INTO memories (user_id, session_id, content, embedding, memory_type, importance)
      VALUES {values}
      ON CONFLICT DO NOTHING
    """, args)
    
    inserted += cur.rowcount
    conn.commit()
    
    console.print(f"  Migrated: {inserted}/{len(memories)}")

console.print(f"🎉 Done! Migrated {inserted} memories")

# Cleanup
cur.close()
conn.close()
duck.close()

## ✅ Complete!